In [1]:
import sys, os, tempfile, timeit, pickle, inspect
from dsc.dsc_io import load_dsc as __load_dsc__, source_dirs as __source_dirs__
import numpy as np
sys.path.append("/gpfs/commons/home/sbanerjee/work/npd/lrma-dsc/dsc/functions")
from comparison_metrics import (
    standardize,
    coupled_procrustes_per_factor_scaling,
    match_latent_dimensions,
    root_mean_squared_error,
    peak_signal_to_noise_ratio,
    adjusted_mutual_information_score
)

In [30]:
DSC_C1BF078C = dict()
DSC_C1BF078C = __load_dsc__(
    ['/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/blockdiag_p/blockdiag_p_4.pkl',
     '/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/gleanr/blockdiag_p_4_identical_1_gleanr_1.rds'])
DSC_REPLICATE = DSC_C1BF078C["DSC_DEBUG"]["replicate"]
DSC_SEED = DSC_C1BF078C["DSC_DEBUG"]["seed"] + 27
F = DSC_C1BF078C['F_est']
Ftrue = DSC_C1BF078C['Ftrue']
L = DSC_C1BF078C['L_est']
Ltrue = DSC_C1BF078C['Ltrue']
labels = DSC_C1BF078C['Ctrue']
for name, func in __source_dirs__(['functions']):
    globals()[name] = func
TIC_C1BF078C = timeit.default_timer()
DSC_SEED += DSC_REPLICATE
import random
random.seed(DSC_SEED)
try:
	import numpy
	numpy.random.seed(DSC_SEED)
except Exception:
	pass

In [31]:
L.shape

(200, 4)

In [87]:
from scipy.linalg import orthogonal_procrustes


def rebalance_to_unit_F(L, F, eps=1e-12, contrib_tol=1e-12):
    """
    Product-preserving rescaling:
        L F.T = (L * ||F_j||) (F_j / ||F_j||).T

    After this, columns of F have norm 1 when nonzero.
    """
    F_norm = np.sqrt(np.sum(F * F, axis=0))
    L_norm = np.sqrt(np.sum(L * L, axis=0))

    L_new = L.copy()
    F_new = F.copy()

    # Columns that can be safely rescaled.
    rescalable = F_norm > eps

    L_new[:, rescalable] = L_new[:, rescalable] * F_norm[rescalable].reshape(1, -1)
    F_new[:, rescalable] = F_new[:, rescalable] / F_norm[rescalable].reshape(1, -1)

    # Truly zero F columns contribute nothing meaningful.
    L_new[:, ~rescalable] = 0.0
    F_new[:, ~rescalable] = 0.0
    
    # Rebalancing impact: 0 means no change; 1 means typical 10x correction.
    # Contribution from each column
    contrib = L_norm * F_norm
    if np.max(contrib) > eps:
        is_contributing = contrib > contrib_tol * np.max(contrib)
    else:
        is_contributing = contrib > eps
    impact_active = rescalable & is_contributing
    
    if np.any(impact_active):
        log_dev = np.log10(np.maximum(F_norm[impact_active], eps))

        weights = contrib[impact_active]
        weights = weights / np.sum(weights)

        impact_log10 = np.sqrt(np.sum(weights * log_dev**2))
    else:
        impact_log10 = np.nan

    return L_new, F_new, impact_log10

def coupled_procrustes_per_factor_scaling_debug(L_true, F_true, L_hat, F_hat, dim_policy="zerofill", eps=1e-8, debug=False):
    """
    Coupled alignment of (L_hat, F_hat) to (L_true, F_true).

    Step 1: find one orthogonal matrix R from F_hat -> F_true:
        min_R || F_true - F_hat R ||_F

    Step 2: after rotation, fit one scale per factor/column:
        d_k = argmin_d || F_true[:,k] - d * (F_hat R)[:,k] ||_2^2

    Then:
        F_aligned = (F_hat R) D
        L_aligned = (L_hat R) D^{-1}

    so that:
        L_aligned @ F_aligned.T == (approximately) L_hat @ F_hat.T
    exactly, except for columns where scale handling hits numerical safeguards.
    """

    F_true_m, F_hat_m = match_latent_dimensions(F_true, F_hat, dim_policy = dim_policy)
    L_true_m, L_hat_m = match_latent_dimensions(L_true, L_hat, dim_policy = dim_policy)
    
    # if factor_scale_non_identifiable(L_true_m, F_true_m, L_hat_m, F_hat_m):
    #     L_hat_m_huge = L_hat_m.copy()
    #     F_hat_m_tiny = F_hat_m.copy()
    #     L_hat_m, F_hat_m = rebalance_to_unit_F(L_hat_m_huge, F_hat_m_tiny)

    # gleanr sometimes produces a single column of zero values
    # We can't rescue with alignment. In fact, procrustes returns error.
    if np.ptp(F_hat_m) == 0:
        k = F_hat_m.shape[1]
        R = np.eye(k) # no rotation, identity matrix
        scales_safe = np.ones(k, dtype=float)
        F_aligned = F_hat_m.copy()
        L_aligned = L_hat_m.copy()
    else:        
        # Shared orthogonal alignment
        R, _ = orthogonal_procrustes(F_hat_m, F_true_m)
    
        F_rot = F_hat_m @ R
        L_rot = L_hat_m @ R
    
        # One scale per aligned column
        if debug:
            print (f"F_hat norm: {np.sqrt(np.sum(F_hat_m  * F_hat_m,  axis = 0))}")
            print (f"F_true norm {np.sqrt(np.sum(F_true_m * F_true_m, axis = 0))}")
        numer = np.sum(F_rot * F_true_m, axis=0)
        denom = np.sum(F_rot * F_rot, axis=0)
        scales = np.ones(F_rot.shape[1], dtype=float)
        if debug:
            print(f"F_rot squared norm (denom): {denom}")
        # valid mask for columns with enough signal
        valid = denom > eps
        if debug:
            print (f"Scale is valid (denom > eps): {valid}")
        scales[valid] = numer[valid] / denom[valid]
        # Clip nearly zero scales for numerical stability. But, keep the sign.
        # Only protect genuinely fitted scales, not padded/invalid columns
        tiny_valid = valid & (np.abs(scales) < eps)
        scales_safe = scales.copy()
        scales_safe[tiny_valid] = np.where(scales_safe[tiny_valid] >= 0, eps, -eps)
    
        F_aligned = F_rot * scales_safe.reshape(1, -1)
        L_aligned = L_rot / scales_safe.reshape(1, -1)
        
    return {
        "L_true": L_true_m,
        "F_true": F_true_m,
        "L_aligned": L_aligned,
        "F_aligned": F_aligned,
        "rotation": R,
        "scales": scales_safe,
    }

def relative_rmse(rmse, original):
    """
    This is the same as relative Frobenius error.
    ||A - B||_F / ||A||_F
    """
    return rmse / np.sqrt(np.mean(original ** 2))

def global_calibration(Z, Z_hat, eps = 1e-12):
    denom = np.sum(Z_hat ** 2)
    true_norm_sq = np.sum(Z ** 2)
    threshold = eps**2 * true_norm_sq
    if denom <= threshold:
        sg = 0.0
        is_zero_norm = True
    else:
        sg = np.sum(Z * Z_hat) / denom
        is_zero_norm = False
    return sg, is_zero_norm

In [89]:
# True signal
Zt = Ltrue @ Ftrue.T

# Raw estimated signal
Za_raw = L @ F.T

# some methods introduce attenuation bias 
# remove only the attenuation bias while preserving row-wise and column-wise structure.
# At zero norm, we force L and F to zero.
sg, is_zero_norm = global_calibration(Zt, Za_raw)
Za = sg * Za_raw
# Assign global calibration to L by convention
L_cal = sg * L
F_cal = np.zeros_like(F) if is_zero_norm else F.copy()

# our simulations use orthonormal factors.
# Rebalance L and F enforcing ||F||_2 = 1
L_bal, F_bal, balancing_impact = rebalance_to_unit_F(L_cal, F_cal)

# Oracle alignment for factor recovery metrics
aligned = coupled_procrustes_per_factor_scaling_debug(Ltrue, Ftrue, L_bal, F_bal, dim_policy="zerofill")
Lt = aligned["L_true"] 
Ft = aligned["F_true"] 
La = aligned["L_aligned"]
Fa = aligned["F_aligned"]
scales = aligned["scales"]

# RMSE
L_rmse = root_mean_squared_error(Lt, La)
F_rmse = root_mean_squared_error(Ft, Fa)
Z_rmse = root_mean_squared_error(Zt, Za)

# Relative RMSE / relative Frobenius error
L_rel_rmse = relative_rmse(L_rmse, Lt)
F_rel_rmse = relative_rmse(F_rmse, Ft)
Z_rel_rmse = relative_rmse(Z_rmse, Zt)

# Optional PSNR; not primary
L_psnr = peak_signal_to_noise_ratio(Lt, La)
F_psnr = peak_signal_to_noise_ratio(Ft, Fa)
Z_psnr = peak_signal_to_noise_ratio(Zt, Za)

# Clustering metrics
MI_raw, adj_MI_raw = adjusted_mutual_information_score(L, labels)
MI_cal, adj_MI_cal = adjusted_mutual_information_score(L_cal, labels)
MI_oracle_aligned, adj_MI_oracle_aligned = adjusted_mutual_information_score(La, labels)

print (
    f"L_rmse: {L_rmse:g}\n"
    f"F_rmse: {F_rmse:g}\n"
    f"Z_rmse: {Z_rmse:g}\n"
    f"L_rel_rmse: {L_rel_rmse:g}\n"
    f"F_rel_rmse: {F_rel_rmse:g}\n"
    f"Z_rel_rmse: {Z_rel_rmse:g}\n"
    f"Global scale: {sg:g}\n"
    f"scales: {scales}\n"
    f"Balancing impact: {balancing_impact}\n"
)

L_rmse: 0.0572988
F_rmse: 0.00823339
Z_rmse: 0.00203422
L_rel_rmse: 0.546745
F_rel_rmse: 0.823339
Z_rel_rmse: 0.613814
Global scale: 1
scales: [0.81065886 0.86054759 0.93101383 0.7995772  0.861939   0.89940428
 1.02495198 0.837437   0.86343582 0.89604336]
Balancing impact: 3.8357332722526367



In [4]:
import glob
def check_L_est_dimension(outdir, file_ext):
    fnames = glob.glob(f"{outdir}/*.{file_ext}")
    dim_dict = dict()
    for f in fnames:
        fbase = os.path.basename(f)
        mres = __load_dsc__([f])
        dim_dict[fbase] = mres['L_est'].shape[1]
    return dim_dict

outdir = "/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/gleanr"
_mres_kdict = check_L_est_dimension(outdir, "rds")

In [5]:
_mres_kdict.values()

dict_values([3, 1, 5, 5, 3, 10, 5, 6, 2, 10, 7, 4, 10, 7, 10, 10, 4, 10, 2, 10, 10, 6, 4, 3, 10, 2, 2, 10, 10, 5, 10, 8, 4, 1, 9, 5, 10, 4, 2, 4, 10, 5, 10, 5, 5, 3, 10, 10, 10, 10, 10, 4, 6, 6, 5, 10, 10, 10, 4, 5, 10, 10, 6, 10, 3, 3, 1, 6, 8, 10, 6, 10, 10, 4, 2, 10, 2, 7, 10, 3, 10, 2, 10, 10, 5, 1, 10, 10, 5, 10, 10, 10, 3, 2, 2, 10, 1, 10, 10, 10, 1, 3, 4, 10, 10, 8, 4, 10, 10, 3, 10, 10, 4, 2, 5, 5, 8, 4, 4, 7, 2, 10, 10, 6, 2, 10, 6, 10, 10, 10, 10, 10, 10, 10, 5, 10, 4, 10, 2, 4, 10, 6, 6, 10, 5, 6, 10, 10, 10, 2, 6, 5, 4, 10, 5, 2, 6, 10, 2, 4, 6, 4, 10, 10, 4, 5, 10, 4, 6, 7, 10, 10, 2, 1, 9, 10, 5, 10, 3, 10, 3, 10, 10, 10, 5, 6, 10, 5, 6, 3])

In [46]:
_select_sims = [fname for fname, k in _mres_kdict.items()]
for f in _select_sims:
    mres = __load_dsc__([os.path.join(outdir, f)])
    if np.any(np.all(mres['L_est'] == mres['L_est'][0:1, :], axis = 0)):
        print(f)
        #print(mres['L_est'])

blockdiag_p_28_identical_1_gleanr_1.rds
blockdiag_p_32_identical_1_gleanr_1.rds


In [54]:
def _matrix_dissimilarity_scores(original, recovered, mask = None, match = 'zerofill'):
    n_orig = original.shape[1]
    n_recv = recovered.shape[1]
    if match == 'clip':
        n = min(n_orig, n_recv)
        X = original[:, :n]
        Y = recovered[:, :n]
    elif match == 'zerofill':
        m = original.shape[0]
        n = max(n_orig, n_recv)
        X = np.zeros((m, n))
        Y = np.zeros((m, n))
        X[:, :n_orig] = original
        Y[:, :n_recv] = recovered
    # gleanr sometimes produces a single column of zero values
    # procrustes requires: Input matrices must contain >1 unique points
    # a matrix has no unique points iff max - min == 0.
    if np.ptp(Y) == 0 :
        m2 = np.sum(np.square(X))
    else:
        R_orig, R_recv, m2 = procrustes(X, Y)
    psnr = peak_signal_to_noise_ratio(R_orig, R_recv, mask)
    return np.sqrt(m2), psnr

In [77]:
sys.path.append("/gpfs/commons/home/sbanerjee/work/npd/lrma-dsc/dsc/functions")

## BEGIN DSC CORE
import numpy as np
from comparison_metrics import matrix_dissimilarity_scores, adjusted_mutual_information_score, mean_squared_error, peak_signal_to_noise_ratio
L_rmse, L_psnr = matrix_dissimilarity_scores(Ltrue, L)
# F_rmse, F_psnr = matrix_dissimilarity_scores(Ftrue, F)
Ztrue = Ltrue @ Ftrue.T
Zrecv = L @ F.T
Ztrue = Ztrue - np.mean(Ztrue, axis = 0, keepdims = True)
Zrecv = Zrecv - np.mean(Zrecv, axis = 0, keepdims = True)
Z_rmse = np.sqrt(mean_squared_error(Ztrue, Zrecv))
Z_psnr = peak_signal_to_noise_ratio(Ztrue, Zrecv)
adj_MI = adjusted_mutual_information_score(L, labels)
## END DSC CORE

ValueError: Input matrices must contain >1 unique points

In [68]:
np.square(L_rmse) / 200

0.004981024159461685

In [75]:
np.sqrt(np.sum(np.square(Ltrue)))

4.100206872020673

In [69]:
mean_squared_error(Ltrue, L)

0.008405807375362704

In [48]:
Z_rmse

0.0026733336396964967

In [49]:
Z_psnr

22.249329802386598

In [50]:
adj_MI

-0.0020696076944895407